In [1]:
import geogridfusion
import pvdeg
import pandas as pd
import pvlib

In [2]:
sw, sm = pvlib.iotools.get_solrad(station="abq", start="2022-01-01", end="2022-01-05")

In [ ]:
geogridfusion.store_single(
    conn=conn,
    weather_df=pvdeg.weather.map_weather(sw),
    meta=pvdeg.weather.map_meta(sm),
    source_name="solrad",
    tmy=False
)

In [3]:
conn = geogridfusion.start()

Starting Postgres subprocess...
PostgreSQL connection established after 2.22 seconds.
postgis already installed


In [ ]:
geogridfusion.initialize_tables(conn)

In [ ]:
client = pvdeg.geospatial.start_dask()

coords = [(45 + i, -115 + k) for i in range(5) for k in range(5)]

geo_weather, geo_meta, failed = pvdeg.weather.weather_distributed(
    database="PVGIS",
    coords=coords
)

client.close()

In [ ]:
geo_meta

In [ ]:
weather, meta = pvdeg.weather.get(
    database="PVGIS",
    id=(45,-115)
)

In [ ]:
geogridfusion.store_single(conn=conn, weather_df=weather, meta=meta, tmy=True, source_res="pvgis")

When we run the cell below, we see that we will get a collsion from the data inserted above.

We want to rewrite this function so it can be done async or with multiprocessing.

In [ ]:
for i in range(25):

    w = geo_weather.isel(gid=i).drop_vars(("gid",)).to_pandas()
    m = geo_meta.iloc[i].to_dict()

    geogridfusion.store_single(conn=conn, weather_df=w, meta=m, tmy=True, source_res="pvgis")

We can easily get one output from what we have written so far. We need to be able to read MANY into dataset form.

In [ ]:
sm

In [4]:
lw, lm = geogridfusion.load_single(conn=conn, latitude=sm["latitude"], longitude=sm["longitude"])

lw

,Year,julian_day,Month,Day,Hour,Minute,decimal_time,solar_zenith,ghi,ghi_flag,...,dhi,dhi_flag,uvb,uvb_flag,uvb_temp,uvb_temp_flag,std_dw_psp,std_direct,std_diffuse,std_uvb
2022-01-01 00:00:00+00:00,2022,1,1,1,0,0,0.000,89.30,2.8,0,...,5.4,0,0.4,0,43.0,0,0.126,0.000,0.316,0.00
2022-01-01 00:01:00+00:00,2022,1,1,1,0,1,0.017,89.44,2.1,0,...,4.8,0,0.4,0,43.0,0,0.252,0.251,0.210,0.03
2022-01-01 00:02:00+00:00,2022,1,1,1,0,2,0.033,89.58,1.4,0,...,4.5,0,0.3,0,43.0,0,0.126,0.000,0.105,0.03
2022-01-01 00:03:00+00:00,2022,1,1,1,0,3,0.050,89.72,1.0,0,...,4.0,0,0.2,0,43.0,0,0.252,0.000,0.105,0.00
2022-01-01 00:04:00+00:00,2022,1,1,1,0,4,0.067,89.84,0.4,0,...,3.7,0,0.2,0,43.0,0,0.000,0.126,0.210,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-01-05 23:55:00+00:00,2022,5,1,5,23,55,23.917,87.89,8.8,0,...,9.9,0,0.8,0,43.1,0,0.882,29.541,0.000,0.03
2022-01-05 23:56:00+00:00,2022,5,1,5,23,56,23.933,88.05,12.2,0,...,9.9,0,0.7,0,43.1,0,0.882,32.432,0.000,0.03
2022-01-05 23:57:00+00:00,2022,5,1,5,23,57,23.950,88.21,12.2,0,...,9.9,0,0.7,0,43.1,0,0.630,3.520,0.000,0.00
2022-01-05 23:58:00+00:00,2022,5,1,5,23,58,23.967,88.37,10.0,0,...,9.9,0,0.6,0,43.1,0,0.756,7.291,0.000,0.02
